# 03 — LayerNorm over GPT-2 small's 768 features

GPT-2 small has **25 LayerNorms**: `ln_1` and `ln_2` in each of the 12 blocks, plus the final `ln_f`. Each normalizes every token's 768-vector and has 768 gains + 768 biases, so 25 × 1,536 = **38,400 parameters** in total. They are tiny in parameter count, but they sit on every path the gradient takes.

You've seen the geometric intuition already (the gradient is a projection that removes the mean direction and the $\hat x$ direction, then divides by σ). Here we turn it into checked code that works on `(B, T, C)`. We'll also build the explicit Jacobian once, to see that the closed-form formula really is the same thing.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

rng = np.random.default_rng(0)

## 1. Forward, per token (last axis only)

$$
\mu = \tfrac1C\textstyle\sum_i x_i,\quad
\sigma = \sqrt{\tfrac1C\sum_i (x_i-\mu)^2 + \epsilon},\quad
\hat x = \dfrac{x-\mu}{\sigma},\quad
y = \gamma\odot\hat x + \beta
$$

LayerNorm normalizes **each token vector separately** over its C = 768 features: for input `(B, T, 768)`, it computes B·T means and B·T standard deviations. (BatchNorm normalizes each feature over the batch; that's not used in transformers.) $\gamma$ ("gain", `g`) and $\beta$ ("bias", `b`) have shape `(C,)` and are shared by every token.

**What to cache for backward:** $\hat x$ and $1/\sigma$ (called `rstd`, "reciprocal std"). Nothing else is needed.

## 2. Backward

**Parameters.** $y = \gamma\odot\hat x+\beta$, and $\gamma,\beta$ are broadcast over all B·T tokens, so sum over them (rule 2):

$$
d\gamma = \sum_{b,t} dy_{b,t}\odot\hat x_{b,t},\qquad d\beta = \sum_{b,t} dy_{b,t}
$$

**Input.** Let $h = dy\odot\gamma = \partial\mathcal{L}/\partial\hat x$. Using $\partial\mu/\partial x_j = 1/C$ and $\partial\sigma/\partial x_j = \hat x_j/C$:

$$
\frac{\partial\hat x_i}{\partial x_j} = \frac1\sigma\Big(\delta_{ij} - \frac1C - \frac{\hat x_i\hat x_j}{C}\Big)
\quad\Longrightarrow\quad
\boxed{dx = \frac{1}{\sigma}\Big(h - \overline{h} - \hat x\;\overline{h\odot\hat x}\Big)}
$$

where the bars are means over the C features of each token.

A detail about $\epsilon$: since $\sigma^2 = \text{var}+\epsilon$, we have $2\sigma\,\partial\sigma = \partial\,\text{var}$, and $\partial\,\text{var}/\partial x_j = \tfrac2C(x_j-\mu)$ (the $\mu$-dependence drops out because $\sum_i(x_i-\mu)=0$). So $\partial\sigma/\partial x_j = \hat x_j/C$ holds **exactly**, $\epsilon$ included. The formula is exact; only the "it's a perfect orthogonal projection" interpretation is approximate when $\epsilon$ matters.

In [ ]:
class LayerNorm:
    def __init__(self, C, eps=1e-5, dtype=np.float64):        # GPT-2 uses eps = 1e-5
        self.eps = eps
        self.params = {"g": np.ones(C, dtype=dtype), "b": np.zeros(C, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, x):                                   # x: (..., C)
        mu = x.mean(axis=-1, keepdims=True)                 # (..., 1)
        xc = x - mu
        var = (xc * xc).mean(axis=-1, keepdims=True)        # biased variance (divide by C), like PyTorch
        self.rstd = 1.0 / np.sqrt(var + self.eps)           # (..., 1)
        self.xhat = xc * self.rstd                          # (..., C)
        return self.xhat * self.params["g"] + self.params["b"]

    def backward(self, dout):                               # dout: (..., C)
        C = dout.shape[-1]
        self.grads["g"] += (dout * self.xhat).reshape(-1, C).sum(axis=0)
        self.grads["b"] += dout.reshape(-1, C).sum(axis=0)
        h = dout * self.params["g"]                         # dL/dxhat
        return self.rstd * (h
                            - h.mean(axis=-1, keepdims=True)
                            - self.xhat * (h * self.xhat).mean(axis=-1, keepdims=True))

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]

In [ ]:
B, T, C = 2, 3, 6
ln = LayerNorm(C)
ln.params["g"][:] = rng.normal(size=C)          # non-trivial gain/bias so they're really tested
ln.params["b"][:] = rng.normal(size=C)
x = rng.normal(size=(B, T, C)) * 3 + 1
G = rng.normal(size=(B, T, C))
f = lambda: np.sum(ln.forward(x) * G)

f()
dx = ln.backward(G)
print("dx rel err:", rel_error(dx, numerical_grad(f, x)))
for name, p, g in ln.parameters():
    print(f"d{name} rel err:", rel_error(g, numerical_grad(f, p)))

### Full check at GPT-2 small's size (C = 768)

LayerNorm is cheap enough to check **every** gradient entry at the real width: B=2, T=4, C=768 is 6,144 inputs, so 12,288 forward passes of one LayerNorm. It takes a few seconds.

In [ ]:
ln768 = LayerNorm(768)
ln768.params["g"][:] = 1 + 0.1 * rng.normal(size=768)
ln768.params["b"][:] = 0.1 * rng.normal(size=768)
x768 = rng.normal(size=(2, 4, 768))
G768 = rng.normal(size=(2, 4, 768))
f768 = lambda: np.sum(ln768.forward(x768) * G768)
f768()
dx768 = ln768.backward(G768)
print("dx rel err (all 6,144 entries):", rel_error(dx768, numerical_grad(f768, x768)))
for name, p, g in ln768.parameters():
    print(f"d{name} rel err (all 768 entries):", rel_error(g, numerical_grad(f768, p)))

## 3. Seeing the projection numerically

The two "blind directions" of LayerNorm are: shifting all features by a constant, and scaling about the mean. The gradient with respect to $x$ must be orthogonal to both, so for every token:

- `dx.sum(-1) ≈ 0` (orthogonal to $\mathbf 1$)
- `(dx * xhat).sum(-1) ≈ 0` (orthogonal to $\hat x$)

In [ ]:
print("sum(dx) per token      :", dx.sum(-1).ravel())
print("sum(dx * xhat) per token:", (dx * ln.xhat).sum(-1).ravel())

## 4. The same thing with an explicit Jacobian (for one token)

For one token, build $J = \frac1\sigma(I - \frac1C\mathbf 1\mathbf 1^\top - \frac1C\hat x\hat x^\top)$, then compute $dx = J^\top h$, and compare with the closed form. This is what your commented-out `backward` was trying to do with `expand_dims`. It works, but it costs $O(C^2)$ memory per token: for GPT-2 small that's 768² = 589,824 numbers per token, and at B = 8, T = 1024 about 4.8 billion numbers (19 GB in float32) for a single LayerNorm. The closed form is $O(C)$ per token.

In [ ]:
x1 = rng.normal(size=C) * 2
G1 = rng.normal(size=C)
ln1 = LayerNorm(C)
ln1.forward(x1)
xhat, rstd = ln1.xhat, ln1.rstd
J = rstd * (np.eye(C) - np.ones((C, C)) / C - np.outer(xhat, xhat) / C)
h = G1 * ln1.params["g"]
print("explicit Jacobian vs closed form:", rel_error(J.T @ h, ln1.backward(G1)))
print("J symmetric:", np.allclose(J, J.T))

## 5. Bonus: RMSNorm (Llama-style decoder-only models)

GPT-2 uses LayerNorm. Most newer decoder-only models use RMSNorm, which skips centering and the bias:

$$
r = \sqrt{\tfrac1C\textstyle\sum_i x_i^2+\epsilon},\quad \hat x = x/r,\quad y=\gamma\odot\hat x
\qquad\Longrightarrow\qquad
dx = \frac1r\Big(h - \hat x\;\overline{h\odot\hat x}\Big),\; h = dy\odot\gamma
$$

It is the LayerNorm formula with the mean-removal term deleted, because RMSNorm is no longer blind to shifts, only to scaling.

In [ ]:
class RMSNorm:
    def __init__(self, C, eps=1e-5):
        self.eps = eps
        self.params = {"g": np.ones(C)}
        self.grads = {"g": np.zeros(C)}
    def forward(self, x):
        self.rrms = 1.0 / np.sqrt((x * x).mean(axis=-1, keepdims=True) + self.eps)
        self.xhat = x * self.rrms
        return self.xhat * self.params["g"]
    def backward(self, dout):
        C = dout.shape[-1]
        self.grads["g"] += (dout * self.xhat).reshape(-1, C).sum(axis=0)
        h = dout * self.params["g"]
        return self.rrms * (h - self.xhat * (h * self.xhat).mean(axis=-1, keepdims=True))

rms = RMSNorm(C)
rms.params["g"][:] = rng.normal(size=C)
f = lambda: np.sum(rms.forward(x) * G)
f()
print("RMSNorm dx rel err:", rel_error(rms.backward(G), numerical_grad(f, x)))
print("RMSNorm dg rel err:", rel_error(rms.grads["g"], numerical_grad(f, rms.params["g"])))

## Summary

```python
# forward caches xhat, rstd
dg = sum over (B,T) of dout * xhat
db = sum over (B,T) of dout
h  = dout * g
dx = rstd * (h - mean(h) - xhat * mean(h * xhat))      # means over the last axis
```

Your forward pass in the script is correct. It just needs to keep `xhat` and `1/std` around (it already stores `mu` and `std`, which is enough to rebuild them).

Next: **04 — Causal self-attention**, the only layer that mixes tokens, and the one with the most intricate backward pass.